In [1]:
from datasets import load_dataset

dataset = load_dataset("lhoestq/conll2003")
dataset["train"][848]

{'id': '848',
 'tokens': ['Dean',
  'Palmer',
  'hit',
  'his',
  '30th',
  'homer',
  'for',
  'the',
  'Rangers',
  '.'],
 'pos_tags': [22, 22, 38, 29, 16, 21, 15, 12, 23, 7],
 'chunk_tags': [11, 12, 21, 11, 12, 12, 13, 11, 12, 0],
 'ner_tags': [1, 2, 0, 0, 0, 0, 0, 0, 3, 0]}

In [2]:
from transformers import AutoModelForTokenClassification, AutoTokenizer

label2id = {
    "O": 0, "B-PER": 1, "I-PER": 2, "B-ORG": 3, "I-ORG": 4,
    "B-LOC": 5, "I-LOC": 6, "B-MISC": 7, "I-MISC": 8
}
id2label = {index: label for label, index in label2id.items()}

model_id = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForTokenClassification.from_pretrained(
    model_id,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

tokens = tokenizer.encode("My name is Maarten")
print(tokenizer.convert_ids_to_tokens(tokens))

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: bert-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

['[CLS]', 'My', 'name', 'is', 'Ma', '##arte', '##n', '[SEP]']


In [3]:
def align_labels(examples):
    token_ids = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)
    labels = examples["ner_tags"]
    updated_labels = []
    for index, label in enumerate(labels):

        # 将词元映射到它们各自的单词
        word_ids = token_ids.word_ids(batch_index=index)
        previous_word_idx = None
        label_ids = []

        for word_idx in word_ids:
            # 新单词的开始
            if word_idx != previous_word_idx:
                previous_word_idx = word_idx
                updated_label = -100 if word_idx is None else label[word_idx]
                label_ids.append(updated_label)
            # 将特殊词元标记为-100
            elif word_idx is None:
                label_ids.append(-100)
            # 如果标签是B-XXX，我们将其改为I-XXX
            else:
                updated_label = label[word_idx]
                if updated_label % 2 == 1:
                    updated_label += 1
                label_ids.append(updated_label)

        updated_labels.append(label_ids)
    token_ids["labels"] = updated_labels
    return token_ids

tokenized = dataset.map(align_labels, batched=True)
tokenized

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags', 'input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 3453
    })
})

In [4]:
# print(f"Original: {example['ner_tags']}")
print(f"Updated: {tokenized['train'][848]['labels']}")

Updated: [-100, 1, 2, 0, 0, 0, 0, 0, 0, 0, 3, 0, -100]


In [6]:
import evaluate
import numpy as np
from transformers import DataCollatorForTokenClassification, TrainingArguments, Trainer

# Load sequential evaluation
seqeval = evaluate.load("seqeval") # 需安装依赖 'pip install seqeval'

def compute_metrics(eval_pred):
    # Create predictions
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=2)

    true_predictions = []
    true_labels = []

    # Document-level iteration
    for prediction, label in zip(predictions, labels):

        # token-level iteration
        for token_prediction, token_label in zip(prediction, label):

            # We ignore special tokens
            if token_label != -100:
                true_predictions.append([id2label[token_prediction]])
                true_labels.append([id2label[token_label]])

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {"f1": results["overall_f1"]}

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

# Training arguments for parameter tuning
training_args = TrainingArguments(
    "ner_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=1,
    weight_decay=0.01,
    save_strategy="epoch",
    report_to="none"
)

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["test"],
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)
trainer.train()
trainer.save_model()

Step,Training Loss
500,0.233101


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [7]:
trainer.evaluate()

Training Loss,Validation Loss,Step,F1
0.233101,0.146051,878,0.898968


{'eval_loss': 0.14605116844177246, 'eval_f1': 0.8989684177114744}

In [9]:
from transformers import pipeline

token_classifier = pipeline ("token-classification", model="ner_model")
entities = token_classifier("My name is Marten and I live in Amsterdam, working for a company called Microsoft. Who is Michael Jackson?")
print(entities)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity': 'B-PER', 'score': np.float32(0.9946502), 'index': 4, 'word': 'Mart', 'start': 11, 'end': 15}, {'entity': 'I-PER', 'score': np.float32(0.9913817), 'index': 5, 'word': '##en', 'start': 15, 'end': 17}, {'entity': 'B-LOC', 'score': np.float32(0.9963955), 'index': 10, 'word': 'Amsterdam', 'start': 32, 'end': 41}, {'entity': 'B-ORG', 'score': np.float32(0.9753651), 'index': 17, 'word': 'Microsoft', 'start': 72, 'end': 81}, {'entity': 'B-PER', 'score': np.float32(0.9554849), 'index': 21, 'word': 'Michael', 'start': 90, 'end': 97}, {'entity': 'I-PER', 'score': np.float32(0.8640685), 'index': 22, 'word': 'Jackson', 'start': 98, 'end': 105}]
